In [1]:
# ==========================================================
# ACTIVIDAD 23. Figuras del documento y tablas para Power BI
# Solo lee resultados ya calculados: no entrena ni modifica
# ningún modelo. Exporta datos AGREGADOS o derivados (regímenes),
# nunca precios originales de Yahoo.
# ==========================================================
import shutil
from pathlib import Path

import joblib
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")
CONTROL, DATOS, MODELOS = RAIZ / "00_control", RAIZ / "01_datos", RAIZ / "02_modelos"
FIGURAS = RAIZ / "03_resultados" / "figuras"
POWER_BI = RAIZ / "03_resultados" / "power_bi"
FIGURAS.mkdir(parents=True, exist_ok=True)
POWER_BI.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Estilo sobrio y común a todas las figuras (ancho de texto A4 ≈ 16 cm)
# ----------------------------------------------------------
COLORES = {"Calma": "#2a78d6", "Impulso": "#eda100", "Estrés": "#e34948"}   # paleta validada (daltonismo)
TINTA, TINTA_SUAVE, REJILLA = "#1f1f1f", "#5a5a5a", "#e6e6e6"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "axes.edgecolor": "#bdbdbd", "axes.labelcolor": TINTA_SUAVE, "xtick.color": TINTA_SUAVE,
    "ytick.color": TINTA_SUAVE, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": REJILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "savefig.dpi": 300, "savefig.bbox": "tight", "figure.facecolor": "white",
})
ANCHO = 6.3                                    # pulgadas
NOMBRES_BLOQUE = {"entrenamiento": "Entrenamiento", "validacion": "Validación", "prueba": "Prueba"}
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]
ETIQUETAS = {"volatilidad_21": "Volatilidad (21 s.)", "cambio_volatilidad": "Cambio de volatilidad",
             "rendimiento_21": "Rendimiento (21 s.)", "caida_maximo_63": "Caída desde máximo (63 s.)",
             "volumen_anormal": "Volumen anormal"}

def coma_decimal(ax, ejes="y"):
    """Números de los ejes con coma decimal, como en el documento en español."""
    formato = mticker.FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))
    if "x" in ejes:
        ax.xaxis.set_major_formatter(formato)
    if "y" in ejes:
        ax.yaxis.set_major_formatter(formato)

def guardar(figura, nombre):
    figura.savefig(FIGURAS / nombre)
    plt.close(figura)
    print("Figura:", nombre)

# ----------------------------------------------------------
# 1. Datos de resultados
# ----------------------------------------------------------
reg = pd.read_parquet(DATOS / "regimenes_k3.parquet")
for col in ["ticker", "grupo", "bloque", "nombre_regimen"]:
    reg[col] = reg[col].astype(str)
muestra = pd.read_csv(CONTROL / "muestra_empresas.csv", dtype={"ticker": str, "cik": str},
                      keep_default_na=False, na_values=[""])
kmeans = joblib.load(MODELOS / "kmeans_k3.joblib")
NOMBRES = kmeans["nombres"]
ORDEN = list(NOMBRES.values())
print("Filas de regímenes:", len(reg))

# % diario de acciones en cada régimen, por grupo de empresas
diario = (reg.groupby(["date", "bloque", "grupo", "nombre_regimen"]).size()
          .rename("acciones").reset_index())
diario["pct"] = diario["acciones"] / diario.groupby(["date", "grupo"])["acciones"].transform("sum") * 100

# ----------------------------------------------------------
# FIGURA 1. Evolución diaria de los regímenes (empresas de entrenamiento)
# ----------------------------------------------------------
fig, ax = plt.subplots(figsize=(ANCHO, 2.8))
ent = diario[diario["grupo"] == "entrenamiento"].pivot_table(
    index=["date", "bloque"], columns="nombre_regimen", values="pct", fill_value=0).reset_index()
for bloque, tramo in ent.groupby("bloque"):
    ax.stackplot(tramo["date"], *[tramo[n] for n in ORDEN], colors=[COLORES[n] for n in ORDEN],
                 linewidth=0)
    ax.text(tramo["date"].iloc[len(tramo) // 2], 101.5, NOMBRES_BLOQUE[bloque], ha="center",
            va="bottom", fontsize=8, color=TINTA_SUAVE)
ax.set_ylim(0, 100)
ax.set_ylabel("% de acciones")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="x", visible=False)
ax.legend([plt.Rectangle((0, 0), 1, 1, color=COLORES[n]) for n in ORDEN], ORDEN,
          loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
guardar(fig, "fig01_evolucion_regimenes.png")

# ----------------------------------------------------------
# FIGURA 2. Perfil de los regímenes (centros del modelo, escala estandarizada)
# ----------------------------------------------------------
modelo = kmeans["modelo"]
centros = pd.DataFrame(modelo.cluster_centers_[np.argsort(modelo.cluster_centers_[:, 0])],
                       columns=VARIABLES, index=ORDEN)
fig, ax = plt.subplots(figsize=(ANCHO, 3.0))
y = np.arange(len(VARIABLES))
alto = 0.26
for i, nombre in enumerate(ORDEN):
    ax.barh(y + (1 - i) * alto, centros.loc[nombre], height=alto - 0.03, color=COLORES[nombre], label=nombre)
ax.axvline(0, color="#9e9e9e", linewidth=0.8)
ax.set_yticks(y, [ETIQUETAS[v] for v in VARIABLES])
ax.invert_yaxis()
ax.set_xlabel("Centro del régimen (desviaciones típicas respecto a la media del entrenamiento)")
ax.grid(axis="y", visible=False)
coma_decimal(ax, "x")
ax.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, -0.2))
guardar(fig, "fig02_perfil_regimenes.png")

# ----------------------------------------------------------
# FIGURA 3. Elección del número de regímenes (k)
# ----------------------------------------------------------
sel = pd.read_csv(MODELOS / "seleccion_k.csv", index_col="k")
fig, ejes = plt.subplots(1, 3, figsize=(ANCHO, 2.2))
paneles = [("Inercia (millones)", [("inercia", "Entrenamiento", 1e6)]),
           ("Silueta", [("silueta_ent", "Entrenamiento", 1), ("silueta_val", "Validación", 1)]),
           ("ARI entrenamiento–validación", [("ari_ent_vs_val", None, 1)])]
for ax, (titulo, series) in zip(ejes, paneles):
    for (columna, etiqueta, escala), estilo in zip(series, ["-", "--"]):
        ax.plot(sel.index, sel[columna] / escala, estilo, color=TINTA, linewidth=1.4,
                marker="o", markersize=3.5, label=etiqueta)
    ax.axvline(3, color=COLORES["Estrés"], linewidth=0.9, linestyle=":")
    ax.set_title(titulo, color=TINTA)
    ax.set_xlabel("k")
    ax.set_xticks(sel.index)
    coma_decimal(ax, "y")
ejes[1].legend(fontsize=7, loc="upper right")
fig.tight_layout(w_pad=1.5)
guardar(fig, "fig03_eleccion_k.png")

# ----------------------------------------------------------
# FIGURA 4. Sincronía diaria del Estrés: empresas de entrenamiento frente a reservadas
# (media móvil de 5 sesiones dentro de cada bloque, solo para que la figura sea legible)
# ----------------------------------------------------------
estres = diario[diario["nombre_regimen"] == "Estrés"].pivot_table(
    index=["date", "bloque"], columns="grupo", values="pct", fill_value=0).reset_index()
fig, ax = plt.subplots(figsize=(ANCHO, 2.6))
for i, (bloque, tramo) in enumerate(estres.groupby("bloque")):
    ax.plot(tramo["date"], tramo["entrenamiento"].rolling(5).mean(), color=TINTA, linewidth=1.0,
            label="Empresas de entrenamiento" if i == 0 else None)
    ax.plot(tramo["date"], tramo["reservada"].rolling(5).mean(), color=COLORES["Estrés"], linewidth=1.0,
            alpha=0.85, label="Empresas reservadas" if i == 0 else None)
ax.set_ylabel("% de acciones en Estrés")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper right")
guardar(fig, "fig04_sincronia_estres.png")

# ----------------------------------------------------------
# FIGURA 5. Estabilidad por celda (K-Means reentrenado frente al modelo definitivo)
# ----------------------------------------------------------
est = pd.read_csv(MODELOS / "generalizacion_estabilidad.csv", index_col="celda")
etiquetas_celda = {c: c.replace("entrenamiento", "Entren.").replace("reservada", "Reservadas")
                   .replace("validacion", "Validación").replace("prueba", "Prueba") for c in est.index}
fig, ax = plt.subplots(figsize=(ANCHO, 2.4))
ax.barh(range(len(est)), est["ari"], color="#7a7a7a", height=0.6)
for i, valor in enumerate(est["ari"]):
    ax.text(valor + 0.01, i, f"{valor:.2f}".replace(".", ","), va="center", fontsize=8, color=TINTA)
ax.set_yticks(range(len(est)), [etiquetas_celda[c] for c in est.index])
ax.invert_yaxis()
ax.set_xlim(0, 1.08)
coma_decimal(ax, "x")
ax.set_xlabel("ARI (empresas / periodo; 1 = misma agrupación)")
ax.grid(axis="y", visible=False)
guardar(fig, "fig05_estabilidad_celdas.png")

# ----------------------------------------------------------
# FIGURA 6. Dirección del precio por grupo: K-Means frente a Gaussian Mixture
# ----------------------------------------------------------
gmm = pd.read_parquet(DATOS / "regimenes_gmm.parquet", columns=["ticker", "date", "gmm"])
gmm["ticker"] = gmm["ticker"].astype(str)
ajuste = reg[(reg["grupo"] == "entrenamiento") & (reg["bloque"] == "entrenamiento")]
ajuste = ajuste.merge(gmm, on=["ticker", "date"], how="left")
grupos = [("K-Means", "regimen", NOMBRES),
          ("Gaussian Mixture", "gmm", {1: "Vol. baja", 2: "Vol. media", 3: "Vol. alta"})]
fig, ejes = plt.subplots(1, 2, figsize=(ANCHO, 2.4), sharey=True)
for ax, (titulo, columna, nombres_g) in zip(ejes, grupos):
    r = ajuste.groupby(columna)["rendimiento_21"]
    subida, caida = r.apply(lambda s: (s > 0.10).mean() * 100), r.apply(lambda s: (s < -0.10).mean() * 100)
    x = np.arange(3)
    ax.bar(x - 0.18, subida.values, width=0.34, color="#1baf7a", label="Mes con subida > 10 %")
    ax.bar(x + 0.18, caida.values, width=0.34, color="#e34948", label="Mes con caída > 10 %")
    ax.set_xticks(x, [nombres_g[i] for i in subida.index])
    ax.set_title(titulo, color=TINTA)
    ax.grid(axis="x", visible=False)
ejes[0].set_ylabel("% de días del grupo")
ejes[0].legend(loc="upper left", fontsize=7)
fig.tight_layout(w_pad=1.5)
guardar(fig, "fig06_direccion_kmeans_gmm.png")

# ----------------------------------------------------------
# 2. Tablas para Power BI (datos agregados o derivados)
# ----------------------------------------------------------
pd.DataFrame({
    "regimen": list(NOMBRES.keys()), "nombre": ORDEN, "color": [COLORES[n] for n in ORDEN],
    "descripcion": [
        "Volatilidad baja y descendente, sin tendencia, cerca del máximo y con poca negociación. Riesgo bajo.",
        "Volatilidad media y en aumento, precio subiendo cerca del máximo y negociación muy alta. Riesgo moderado ligado a subidas.",
        "Volatilidad muy alta, pérdidas fuertes en el mes y caída profunda desde el máximo. Riesgo alto."],
}).to_csv(POWER_BI / "dim_regimen.csv", index=False)

diario.rename(columns={"date": "fecha", "nombre_regimen": "regimen"}).to_csv(
    POWER_BI / "regimen_diario.csv", index=False)

perfiles = (reg.groupby(["grupo", "bloque", "nombre_regimen"])[VARIABLES].median()
            .join(reg.groupby(["grupo", "bloque", "nombre_regimen"]).size().rename("filas")).reset_index())
perfiles["pct_filas"] = perfiles["filas"] / perfiles.groupby(["grupo", "bloque"])["filas"].transform("sum") * 100
perfiles.rename(columns={"nombre_regimen": "regimen"}).to_csv(POWER_BI / "perfiles_regimen.csv", index=False)

empresas = (pd.crosstab([reg["ticker"], reg["bloque"]], reg["nombre_regimen"], normalize="index") * 100)
empresas = empresas.reindex(columns=ORDEN).add_prefix("pct_").reset_index()
empresas["regimen_predominante"] = empresas[[f"pct_{n}" for n in ORDEN]].idxmax(axis=1).str[4:]
empresas = empresas.merge(muestra[["ticker", "grupo", "estrato", "division_sic"]], on="ticker", how="left")
empresas.rename(columns={"pct_Estrés": "pct_Estres"}).to_csv(POWER_BI / "empresas_regimen.csv", index=False)

reg[["ticker", "date", "grupo", "bloque", "nombre_regimen"]].rename(
    columns={"date": "fecha", "nombre_regimen": "regimen"}).to_csv(POWER_BI / "accion_dia_regimen.csv", index=False)

for archivo in ["seleccion_k.csv", "generalizacion_estabilidad.csv", "generalizacion_sincronia.csv",
                "regimenes_trimestrales.csv", "comparacion_kmeans_gmm.csv"]:
    shutil.copy(MODELOS / archivo, POWER_BI / archivo)

print("\nTablas para Power BI:")
for archivo in sorted(POWER_BI.glob("*.csv")):
    print(f"  {archivo.name:36s} {archivo.stat().st_size / 1e6:8.2f} MB")

Mounted at /content/drive
Filas de regímenes: 2426275
Figura: fig01_evolucion_regimenes.png
Figura: fig02_perfil_regimenes.png
Figura: fig03_eleccion_k.png
Figura: fig04_sincronia_estres.png
Figura: fig05_estabilidad_celdas.png
Figura: fig06_direccion_kmeans_gmm.png

Tablas para Power BI:
  accion_dia_regimen.csv                 113.14 MB
  comparacion_kmeans_gmm.csv               0.00 MB
  dim_regimen.csv                          0.00 MB
  empresas_regimen.csv                     0.61 MB
  generalizacion_estabilidad.csv           0.00 MB
  generalizacion_sincronia.csv             0.00 MB
  perfiles_regimen.csv                     0.00 MB
  regimen_diario.csv                       0.58 MB
  regimenes_trimestrales.csv               0.00 MB
  seleccion_k.csv                          0.00 MB
